In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [2]:


# ─── Función objetivo ───────────────────────────────────────────────────────
def funcion(x, y):
    sum1 = x**2 * (4 - 2.1*x**2 + x**4/3.0)
    sum2 = x * y
    sum3 = y**2 * (-4 + 4*y**2)
    return sum1 + sum2 + sum3

# ─── Búsqueda por grilla (programación estructurada) ────────────────────────
def busqueda_grilla(pasos=1000):
    """
    Evalúa la función en una grilla uniforme de 'pasos x pasos' puntos
    dentro del dominio y devuelve el mínimo encontrado.
    """
    mejor_x     = None
    mejor_y     = None
    mejor_valor = float('inf')

    xs = np.linspace(-2.0, 2.0, pasos)   # dominio en x
    ys = np.linspace(-1.0, 1.0, pasos)   # dominio en y

    for x in xs:
        for y in ys:
            valor = funcion(x, y)
            if valor < mejor_valor:
                mejor_valor = valor
                mejor_x     = x
                mejor_y     = y

    return mejor_x, mejor_y, mejor_valor

# ─── Ejecución ──────────────────────────────────────────────────────────────
inicio_pe = time.time()
x_pe, y_pe, val_pe = busqueda_grilla(pasos=1000)
tiempo_pe = time.time() - inicio_pe


print(f"  x óptimo : {x_pe:.6f}")
print(f"  y óptimo : {y_pe:.6f}")
print(f"  f(x,y)   : {val_pe:.6f}")
print(f"  Tiempo   : {tiempo_pe:.4f} s")


  x óptimo : -0.090090
  y óptimo : 0.711712
  f(x,y)   : -1.031621
  Tiempo   : 0.9381 s


In [5]:
from random import random

# ─── Función auxiliar de número aleatorio uniforme ──────────────────────────
def aleatorio(inf, sup):
    return random() * (sup - inf) + inf

# ─── Clase Partícula ─────────────────────────────────────────────────────────
class Particula:
    # Parámetros comunes a todas las partículas
    inercia   = 1.4
    cognitiva = 2.0
    social    = 2.0

    # Límites del espacio de soluciones
    infx = -2.0
    supx =  2.0
    infy = -1.0
    supy =  1.0

    # Factor de ajuste de la velocidad inicial
    ajusteV = 100.0

    def __init__(self):
        """Crea una partícula dentro de los límites indicados."""
        self.x  = aleatorio(Particula.infx, Particula.supx)
        self.y  = aleatorio(Particula.infy, Particula.supy)
        self.vx = aleatorio(Particula.infx / Particula.ajusteV,
                            Particula.supx / Particula.ajusteV)
        self.vy = aleatorio(Particula.infy / Particula.ajusteV,
                            Particula.supy / Particula.ajusteV)
        # Mejor posición local
        self.xLoc      = self.x
        self.yLoc      = self.y
        self.valorLoc  = funcion(self.x, self.y)

    def actualizaVelocidad(self, xGlob, yGlob):
        """Actualiza la velocidad de la partícula."""
        cogX = Particula.cognitiva * random() * (self.xLoc - self.x)
        socX = Particula.social    * random() * (xGlob    - self.x)
        self.vx = Particula.inercia * self.vx + cogX + socX

        cogY = Particula.cognitiva * random() * (self.yLoc - self.y)
        socY = Particula.social    * random() * (yGlob    - self.y)
        self.vy = Particula.inercia * self.vy + cogY + socY

    def actualizaPosicion(self):
        """Actualiza la posición y aplica límites del dominio."""
        self.x = self.x + self.vx
        self.y = self.y + self.vy

        # Mantenerse dentro del espacio de soluciones
        self.x = max(self.x, Particula.infx)
        self.x = min(self.x, Particula.supx)
        self.y = max(self.y, Particula.infy)
        self.y = min(self.y, Particula.supy)

        # Si es inferior a la mejor local, actualizar
        valor = funcion(self.x, self.y)
        if valor < self.valorLoc:
            self.xLoc     = self.x
            self.yLoc     = self.y
            self.valorLoc = valor

# ─── Función principal PSO ───────────────────────────────────────────────────
def enjambreParticulas(particulas, iteraciones, reduccionInercia):
    """
    Mueve un enjambre de partículas durante las iteraciones indicadas.
    Devuelve las coordenadas y el valor del mínimo obtenido.
    """
    historial = []   # para graficar convergencia

    # Registra la mejor posición global y su valor
    mejorParticula = min(particulas, key=lambda p: p.valorLoc)
    xGlob      = mejorParticula.xLoc
    yGlob      = mejorParticula.yLoc
    valorGlob  = mejorParticula.valorLoc

    for _ in range(iteraciones):
        # Actualiza velocidad y posición de cada partícula
        for p in particulas:
            p.actualizaVelocidad(xGlob, yGlob)
            p.actualizaPosicion()

        # Actualiza mínimo global
        mejorParticula = min(particulas, key=lambda p: p.valorLoc)
        if mejorParticula.valorLoc < valorGlob:
            xGlob     = mejorParticula.xLoc
            yGlob     = mejorParticula.yLoc
            valorGlob = mejorParticula.valorLoc

        historial.append(valorGlob)

        # Reduce la inercia de las partículas
        Particula.inercia *= reduccionInercia

    return xGlob, yGlob, valorGlob, historial

# ─── Parámetros del problema ─────────────────────────────────────────────────
nParticulas      = 10
iteraciones      = 100
redInercia       = 0.9
Particula.inercia = 1.4   # reiniciar inercia antes de cada ejecución

# ─── Ejecución PSO ───────────────────────────────────────────────────────────
import numpy as np
inicio_pso = time.time()
particulas  = [Particula() for i in range(nParticulas)]
x_pso, y_pso, val_pso, historial = enjambreParticulas(particulas, iteraciones, redInercia)
tiempo_pso  = time.time() - inicio_pso


print(f"  x óptimo : {x_pso:.6f}")
print(f"  y óptimo : {y_pso:.6f}")
print(f"  f(x,y)   : {val_pso:.6f}")
print(f"  Tiempo   : {tiempo_pso:.6f} s")


  x óptimo : -0.089842
  y óptimo : 0.712656
  f(x,y)   : -1.031628
  Tiempo   : 0.004302 s


In [7]:
# ─── Tabla comparativa ───────────────────────────────────────────────────────
print(f"{'Criterio':<30} {'Prog. Estructurada':>16} {'PSO':>13}")
print(f"{'x óptimo':<30} {x_pe:>16.6f} {x_pso:>13.6f}")
print(f"{'y óptimo':<30} {y_pe:>16.6f} {y_pso:>13.6f}")
print(f"{'f(x,y) mínimo':<30} {val_pe:>16.6f} {val_pso:>13.6f}")
print(f"{'Tiempo de ejecución (s)':<30} {tiempo_pe:>16.4f} {tiempo_pso:>13.6f}")
print(f"{'Evaluaciones de función':<30} {'1,000,000':>16} {'~1,000':>13}")

error = abs(val_pe - val_pso)
print(f"\nDiferencia en f(x,y): {error:.8f}")
print(f"Aceleración PSO vs Grilla: {tiempo_pe/tiempo_pso:.1f}x más rápido")

Criterio                       Prog. Estructurada           PSO
x óptimo                              -0.090090     -0.089842
y óptimo                               0.711712      0.712656
f(x,y) mínimo                         -1.031621     -1.031628
Tiempo de ejecución (s)                  0.9381      0.004302
Evaluaciones de función               1,000,000        ~1,000

Diferencia en f(x,y): 0.00000777
Aceleración PSO vs Grilla: 218.1x más rápido
